# B3-J1 Atelier 1 -- PySpark + `.explain()`

**Cours** : Big Data B3 -- Jour 1 (20 mai 2026)
**Duree** : 55 min (15 min papier + 30 min PySpark + 10 min debrief)
**Public** : B3 fullstack (B2 valide)

---

## Objectifs

Apres le word count B2, on va voir **sous le capot de Spark** :

1. Comprendre la **lazy evaluation** (les transformations ne s'executent pas tout de suite)
2. Lire un plan d'execution avec `.explain()`
3. Identifier le **DAG** (Directed Acyclic Graph) que Spark construit
4. Comparer Pandas vs PySpark sur un dataset reel
5. Toucher du doigt l'impact de `.cache()`

> Upgrade B3 vs B2 : on ne se contente plus du resultat. On regarde **comment** Spark optimise la requete.

## Phase 1 -- Exercice MapReduce papier (15 min)

> **Cette phase se fait au tableau, hors notebook.**
>
> Voir le document `exercice-mapreduce-papier.md`.

Rappel rapide :

1. La classe est divisee en 3 groupes (= 3 machines)
2. Chaque groupe recoit **1 paragraphe** et compte les mots
3. On consolide les comptages au tableau

Vous venez de faire :
- **MAP** : chaque groupe traite sa partie en parallele
- **SHUFFLE** : on regroupe les mots identiques au tableau
- **REDUCE** : on additionne les comptages

On va maintenant refaire **exactement la meme chose** avec PySpark.

## Phase 2 -- Setup PySpark sur Colab

PySpark n'est pas pre-installe sur Colab. L'install prend ~30 secondes.

In [ ]:
# Installation PySpark (silencieuse)
!pip install pyspark -q

import time
import pandas as pd
from collections import Counter
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col, lower, desc

# Creer la session Spark
spark = (
    SparkSession.builder
    .appName("B3-J1-Atelier1")
    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print(f"Spark UI       : {spark.sparkContext.uiWebUrl}")
print(f"Master         : {spark.sparkContext.master}")

## Warm-up : word count en Pandas

Meme texte que l'exercice papier (les 3 paragraphes consolides). On compte avec `collections.Counter`.

In [ ]:
# Texte = les 3 paragraphes de l'exercice papier
texte = """
La data est partout. Chaque jour des milliards de donnees sont generees
par les capteurs les telephones et les applications. La data transforme
les entreprises et les gouvernements.

Les entreprises utilisent la data pour prendre des decisions. Les capteurs
IoT generent des donnees en continu. Les applications collectent des
donnees sur les utilisateurs.

Le Big Data permet d analyser la data a grande echelle. Les donnees
massives necessitent des outils specialises. La data science transforme
les donnees en valeur.
"""

# A completer : faire le word count avec collections.Counter
mots = texte.lower().split()
comptage_pandas = Counter(mots)  # A completer si vous voulez tester une autre approche

print(f"Nombre total de mots : {len(mots)}")
print(f"Nombre de mots uniques : {len(comptage_pandas)}")
print("\nTop 10 :")
for mot, count in comptage_pandas.most_common(10):
    print(f"  {mot:<20} {count}")

## Bascule PySpark : meme code, autre moteur

On va refaire le meme word count en Spark. Surprise : **certaines lignes ne vont rien executer du tout**.

C'est la **lazy evaluation** : Spark accumule un plan, et n'execute que quand on lui demande un resultat (action).

In [ ]:
# Version RDD (style B2)
# Etape 1 : paralleliser les mots
rdd = spark.sparkContext.parallelize(texte.lower().split())
print(f"Nombre de partitions : {rdd.getNumPartitions()}")

# Etape 2 : MAP -- chaque mot devient (mot, 1)
# A completer : appliquer un .map() qui transforme chaque mot en tuple (mot, 1)
mapped = rdd.map(lambda mot: (mot, 1))

# Etape 3 : REDUCE -- additionner les 1 par cle
# A completer : utiliser reduceByKey
reduced = mapped.reduceByKey(lambda a, b: a + b)

# Etape 4 : trier par count decroissant et prendre les 10 premiers
top10 = reduced.sortBy(lambda x: -x[1]).take(10)

print("\n=== SPARK RDD ===")
for mot, count in top10:
    print(f"  {mot:<20} {count}")

In [ ]:
# Version DataFrame (recommandee en pratique)
df = spark.createDataFrame([(texte,)], ["texte"])

# A completer : ecrire la requete DataFrame qui :
# 1. transforme le texte en minuscules puis split par espace
# 2. "explode" la colonne pour avoir 1 ligne par mot
# 3. filtre les chaines vides
# 4. groupBy("mot").count() puis trie par count desc
comptage_df = (
    df.select(explode(split(lower(col("texte")), "\\s+")).alias("mot"))
      .filter(col("mot") != "")
      .groupBy("mot")
      .count()
      .orderBy(desc("count"))
)

print("=== SPARK DATAFRAME ===")
comptage_df.show(10, truncate=False)

## UPGRADE B3 -- `.explain()` : sous le capot

Maintenant la nouveaute. Au lieu de juste regarder le **resultat**, on demande a Spark :
**"montre-moi le plan d'execution que tu as construit."**

Spark expose 4 niveaux de plans :
- **parsed** : ce qu'il a lu de votre code
- **analyzed** : apres resolution des types et des colonnes
- **optimized** : apres passage du Catalyst optimizer
- **physical** : ce qui va reellement tourner (avec les noeuds Exchange, HashAggregate, etc.)

In [ ]:
# Plan d'execution simple (physical par defaut)
print("=== PHYSICAL PLAN ===")
comptage_df.explain()

In [ ]:
# Plan d'execution complet (les 4 niveaux)
print("=== ALL PLANS (extended) ===")
comptage_df.explain(extended=True)

## Comment lire ce plan ?

Quelques noeuds qu'on retrouve souvent :

- **`Project`** : selection de colonnes / calcul d'expressions (l'equivalent du `SELECT` SQL)
- **`Filter`** : un `WHERE`
- **`HashAggregate`** : agregation (le `GROUP BY` -- `count`, `sum`, `mean`)
- **`Exchange`** : **redistribution des donnees entre executors** (= shuffle). C'est ce qui coute cher.
- **`Sort`** : tri
- **`Generate explode`** : notre `explode()` qui transforme 1 ligne en N

**Bonne lecture du plan** : on le lit **de bas en haut** (les feuilles = sources, le sommet = resultat final).

**Pourquoi on regarde ca ?** Parce que c'est ce que vous lirez sur un vrai job Spark en production quand il prend 3h au lieu de 5 min. Le coupable est presque toujours un `Exchange` mal place.

## Lazy evaluation : la demo qui pique

On va creer une chaine de transformations **sans rien executer**. Vous allez voir : tant qu'on n'appelle pas une **action** (`.show()`, `.count()`, `.collect()`...), il ne se passe absolument rien.

In [ ]:
# Etape 1 : on construit une serie de transformations sur un gros dataset bidon
import time

t0 = time.time()

# A completer : creer un DataFrame Spark de 10 millions de lignes
# Indice : spark.range(10_000_000) cree une colonne 'id' de 0 a 9_999_999
big_df = spark.range(10_000_000)

# Chaine de transformations -- aucune ne va s'executer
transformed = (
    big_df
    .filter(col("id") % 2 == 0)             # transformation
    .withColumn("double", col("id") * 2)    # transformation
    .filter(col("double") > 1000)           # transformation
    .select("id", "double")                 # transformation
)

t1 = time.time()
print(f"Construction du plan : {t1 - t0:.4f}s")
print("Aucune ligne n'a ete traitee -- Spark a juste enregistre le plan.")

In [ ]:
# Maintenant on appelle une ACTION -- la magie opere
t0 = time.time()
transformed.show(5)
t1 = time.time()
print(f"\nExecution effective : {t1 - t0:.2f}s")
print("Spark a optimise le plan ET execute les transformations en une seule passe.")

## Spark UI : voir le DAG

Spark expose une UI web qui montre :
- les **jobs** lances
- les **stages** et tasks
- le **DAG** (graph des operations)
- les metriques (shuffle, memoire, GC...)

Sur Colab, l'UI tourne sur le port 4040 de la VM. Pour y acceder, deux options :

1. **Tunnel ngrok** (necessite un compte gratuit ngrok et la cle d'auth)
2. **Screenshot statique** que le formateur projette

Code pour ngrok (optionnel) :
```python
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token("VOTRE_TOKEN")
public_url = ngrok.connect(4040)
print(public_url)
```

**Plan B** : on regarde un screenshot du Spark UI projete par le formateur. L'essentiel est de comprendre que le DAG decoupe le job en **stages** separes par des `Exchange` (shuffles).

In [ ]:
# Afficher le lien (peut etre inaccessible depuis votre navigateur sur Colab)
print(f"Spark UI (local Colab) : {spark.sparkContext.uiWebUrl}")
print("\nSi le lien n'ouvre rien : on regarde le screenshot projete.")

## Dataset reel : Spotify

On charge un CSV public depuis GitHub raw et on refait un word count, cette fois sur les **titres de morceaux** (`track_name`).

> Si l'URL ne repond pas, on a un fallback synthetique.

In [ ]:
# A completer si besoin : remplacer l'URL si elle n'est plus accessible
URL_DATASET = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"

import pandas as pd

try:
    pdf = pd.read_csv(URL_DATASET)
    print(f"Dataset Spotify charge : {pdf.shape}")
except Exception as e:
    print(f"URL inaccessible ({e}). Fallback synthetique.")
    import numpy as np
    np.random.seed(0)
    titres = ["Love Story", "Bad Guy", "Dance Monkey", "Shape of You", "Old Town Road",
             "Despacito", "Believer", "Thunder", "Sunflower", "Rockstar"]
    pdf = pd.DataFrame({
        "track_name": np.random.choice(titres, 20_000),
        "track_artist": np.random.choice(["Artist A", "Artist B", "Artist C"], 20_000),
        "playlist_genre": np.random.choice(["pop", "rock", "rap", "latin", "edm", "r&b"], 20_000),
        "track_popularity": np.random.randint(0, 100, 20_000),
    })
    print(f"Fallback synthetique : {pdf.shape}")

pdf.head()

In [ ]:
# Charger dans Spark
sdf = spark.createDataFrame(pdf[["track_name", "playlist_genre", "track_popularity"]])
print(f"Spark DataFrame : {sdf.count()} lignes")
sdf.printSchema()
sdf.show(5, truncate=False)

In [ ]:
# Word count sur les titres de morceaux
wc_tracks = (
    sdf.select(explode(split(lower(col("track_name")), "\\s+")).alias("mot"))
       .filter(col("mot") != "")
       .groupBy("mot")
       .count()
       .orderBy(desc("count"))
)

wc_tracks.show(15, truncate=False)
print("\nPlan d'execution :")
wc_tracks.explain()

## Pandas vs PySpark : le timer

On chronometre le **meme** word count en Pandas et en Spark sur le dataset reel.

In [ ]:
# A completer : chronometrer la version PANDAS (utiliser time.time())
t0 = time.time()
mots_pdf = pdf["track_name"].fillna("").str.lower().str.split().explode()
mots_pdf = mots_pdf[mots_pdf != ""]
top_pdf = mots_pdf.value_counts().head(15)
t_pandas = time.time() - t0

print("=== PANDAS ===")
print(top_pdf)
print(f"\nTemps Pandas : {t_pandas:.3f}s")

In [ ]:
# A completer : chronometrer la version SPARK
t0 = time.time()
top_sdf = wc_tracks.limit(15).collect()
t_spark = time.time() - t0

print("=== SPARK ===")
for row in top_sdf:
    print(f"  {row['mot']:<20} {row['count']}")
print(f"\nTemps Spark  : {t_spark:.3f}s")
print(f"Ratio Spark/Pandas : {t_spark / t_pandas:.1f}x")

## Discussion : qui gagne ?

Sur ce dataset (~30 000 lignes) : **Pandas gagne tres largement** (souvent 10x a 50x plus rapide).

Pourquoi ?
- Spark a un **overhead de demarrage** (creer un DAG, distribuer, collecter les resultats)
- Pandas tourne en C, en RAM, sur 1 coeur, sans coordination

**A quelle taille Spark prend l'avantage ?**

- En general autour de **quelques Go** (au-dela de la RAM disponible sur 1 machine)
- Sur 1 To : Pandas plante, Spark passe (sur un cluster a 20 machines, ca prend ~30 min)

Regle pragmatique :

| Volume | Outil |
|---|---|
| < 1 Go | Pandas |
| 1-100 Go | Pandas (sample) OU Spark local OU Polars |
| > 100 Go | Spark sur cluster |
| > 1 To | Spark, Dask, ou BigQuery |

## Bonus : `.cache()` et impact perf

Si vous reutilisez plusieurs fois le meme DataFrame, **mettez-le en cache**. Spark recalcule tout sinon (lazy + pas de memoire par defaut).

In [ ]:
# Refaire la requete 3 fois SANS cache
t0 = time.time()
for _ in range(3):
    wc_tracks.count()
t_no_cache = time.time() - t0
print(f"3 executions SANS cache : {t_no_cache:.2f}s")

# Refaire la requete 3 fois AVEC cache
wc_cached = wc_tracks.cache()
wc_cached.count()  # premier appel = warm-up du cache

t0 = time.time()
for _ in range(3):
    wc_cached.count()
t_with_cache = time.time() - t0
print(f"3 executions AVEC cache : {t_with_cache:.2f}s")

print(f"\nGain : x{t_no_cache / max(t_with_cache, 0.001):.1f}")

In [ ]:
# Nettoyage
wc_cached.unpersist()
spark.stop()
print("Session Spark fermee.")

## Recap -- ce que vous emportez

1. **Lazy evaluation** : les transformations (filter, select, map...) ne s'executent pas. Seules les **actions** (show, count, collect, write) declenchent l'execution.
2. **`.explain()`** : affiche le plan que Spark va executer. Indispensable pour diagnostiquer un job lent. Cherchez les `Exchange` (= shuffle = cher).
3. **DAG** : Spark decoupe votre code en stages separes par des shuffles. C'est ce qu'on voit dans la Spark UI.
4. **Pandas vs Spark** : sur petit dataset, Pandas gagne. Spark prend l'avantage quand ca ne tient plus en RAM ou qu'il y a un cluster.
5. **`.cache()`** : si vous reutilisez un DataFrame, mettez-le en cache. Sinon Spark recalcule tout.

## A chercher chez vous (2 questions pour J2)

1. **Qu'est-ce qu'un shuffle exactement ?** Pourquoi c'est l'operation la plus couteuse de Spark ? Quels operateurs en provoquent un (`groupBy`, `join`, `orderBy`...) ?
2. **Qu'est-ce qu'un broadcast join ?** Quand est-ce que Spark le declenche, et comment le forcer manuellement avec `broadcast()` ?

Ces deux notions sont la base de l'optimisation Spark. Reponses (sommaires) en debut de J2.